In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib.style
import matplotlib as mpl
import matplotlib.colors as colors
import matplotlib.cm as cmx
mpl.style.use('classic')

import baraffe_tables
from baraffe_tables.table_search import baraffe_table_search

from astropy.table import Table
from astropy import units as u
from astropy.constants import G

import mesa_helper as mh
import os
import shutil

import time

%matplotlib inline

In [ ]:
mJtomSun = u.jupiterMass.to(u.solMass)
mJtoGrams = u.jupiterMass.to(u.g)

rJtorSun = u.jupiterRad.to(u.solRad)
rSuntorJ = u.solRad.to(u.jupiterRad)
rJtoCm = u.jupiterRad.to(u.cm)

#print(mJtomSun)
print(5.0*mJtoGrams)
#print(rJtorSun)
print(2*rJtoCm)


In [ ]:
# rename all existing files to include the initial radius
'''
init_dir = "/Users/emily/Documents/astro/giant_planets/MESA_EoS/thermo_consistency/MESA_runs/make_initial_models"
modfiles = [f for f in os.listdir(init_dir) if os.path.isfile(os.path.join(init_dir, f)) and ".mod" in f]
profilefiles = [f for f in os.listdir(init_dir) if os.path.isfile(os.path.join(init_dir, f)) and ".profile" in f]

for f in profilefiles:
    print(f)
    new_fname = f.split("Mj")[0] + "Mj_2.0_Rj" + f.split("Mj")[1]
    print(new_fname)
    os.system('mv {0}/{1} {0}/{2}'.format(init_dir,f,new_fname))
'''
'''
for f in profilefiles:
    print(f)
    new_fname = f.split("MESAdefault")[0] + "MESA_default" + f.split("MESAdefault")[1]
    print(new_fname)
    os.system('mv {0}/{1} {0}/{2}'.format(init_dir,f,new_fname))
'''

"""
for i, eos in enumerate(eos_names):
    print(eos)
    this_eos_dir = '/Users/emily/Documents/astro/giant_planets/MESA_EoS/thermo_consistency/MESA_runs/eos_' + eos
    os.chdir(this_eos_dir)

    modfiles = [f for f in os.listdir(".") if os.path.isfile(os.path.join(".", f)) and ".mod" in f]
    profilefiles = [f for f in os.listdir(".") if os.path.isfile(os.path.join(".", f)) and ".terminationprofile" in f]
    historyfiles = [f for f in os.listdir(".") if os.path.isfile(os.path.join(".", f)) and ".history" in f]
    logfiles = [f for f in os.listdir("./LOGS") if os.path.isfile(os.path.join("./LOGS", f)) and ".data" in f]
    
    print(len(modfiles),len(profilefiles),len(historyfiles),len(logfiles))
    '''
    for f in historyfiles:
        print(f)
        new_fname = f.split("Mj")[0] + "Mj_2.0_Rj" + f.split("Mj")[1]
        print(new_fname)

        os.system('mv ./{0} ./{1}'.format(f,new_fname))
    '''

    for f in logfiles:
        print(f)
        new_fname = f.split("Mj")[0] + "Mj_2.0_Rj" + f.split("Mj")[1]
        print(new_fname)

        os.system('mv ./LOGS/{0} ./LOGS/{1}'.format(f,new_fname))
"""

In [ ]:
s_arr = np.array((1,2,5))
sf_arr = np.array((0.05,0.1,0.25))

# eos names
eos_names = ['MESA_default','CD21+AQUA_orig_meos_implementation','CMS19_orig_1st_implementation','CMS19_orig_meos_implementation']

for s in s_arr:
    eos_names.append('CMS19_TC_s={0}'.format(s))
    for sf in sf_arr:
        eos_names.append('CMS19_control_s={0}_sf={1}'.format(s,sf))
print(eos_names)
print(len(eos_names))


#comps = ['00z90x','protosolar']
#comps = ['platowg']
comps = ['protosolar']
comp_profiles = ['uniform','exponential']

#Minit = np.array((20.0,10.0,7.5,6.0,5.0,4.0,3.0,2.0,1.0))
Minit = np.array((10.0,5.0,1.0))
#Minit = np.atleast_1d(np.array((1.5)))

Rinit = np.atleast_1d(np.array((2.0,5.0)))

Sinit = np.atleast_1d(np.array((8.0)))

age = 5.0e9
agestr = str(np.round(age/1.e9, 1))
print(agestr)

atm='ttau'

In [ ]:
restart = False

for i, eos in enumerate(eos_names):
    print(eos)
    this_eos_dir = '/Users/emily/Documents/astro/giant_planets/MESA_EoS/thermo_consistency/MESA_runs/eos_' + eos
    os.chdir(this_eos_dir)

    if restart is True:
        ! rm /Users/emily/mesa-24.08.1/data/eosDT_data/cache/*planetblend*
        ! rm /Users/emily/mesa-24.08.1/data/eosDT_data/cache/*CD21+AQUA*

        ! rm ./photos/*
        ! rm ./*.mod
        ! rm ./*.profile
        ! rm ./*.history
        ! rm ./*_consistency.dat
        ! rm ./*.terminationprofile
        ! rm ./LOGS/*

# create initial models

In [ ]:
os.chdir("/Users/emily/Documents/astro/giant_planets/MESA_EoS/thermo_consistency/MESA_runs/make_initial_models")

# eos names
#create_eos_names = ['MESA_default','CD21+AQUA_orig_meos_implementation','CMS19_orig_1st_implementation','CMS19_orig_meos_implementation']
'''
for s in s_arr:
    create_eos_names.append('CMS19_TC_s={0}'.format(s))
    for sf in sf_arr:
        create_eos_names.append('CMS19_control_s={0}_sf={1}'.format(s,sf))
'''

#create_eos_names = ['MESA_default','CD21+AQUA_orig_meos_implementation','CMS19_orig_meos_implementation','CMS19_TC_s=1','CD21_Y0275']

#NOTE: for the below to work, have to modify run_star_extras to import custom_eos_onetable rather than custom_eos
create_eos_names = ['CD21_Y0275']

for i, eos in enumerate(create_eos_names):
    if eos=='MESA_default':
        pass
    else:
        if 'CD21+AQUA' in eos or 'Y0275' in eos:
            eos_str = "mesa-CD21+AQUA_"
                        
        elif 'TC' in eos or 'control' in eos:
            eos_name_stem = eos.split("CMS19_")[1] + "_"
            eos_str = "mesa-planetblend-" + eos_name_stem
                        
        else:
            eos_str = "mesa-planetblend_"

        ! rm /Users/emily/mesa-24.08.1/data/eosDT_data/cache/*planetblend*
        ! rm /Users/emily/mesa-24.08.1/data/eosDT_data/cache/*CD21+AQUA*
        ! rm ./src/data/*

        eos_dir = '/Users/emily/Documents/astro/giant_planets/MESA_EoS/thermo_consistency/MESA_runs/eos_' + eos
        os.system('ln -s {0}/src/data/* ./src/data/'.format(eos_dir))

    
    for j, comp in enumerate(comps):
    
        if comp == 'protosolar':
            X = 0.706
            Z = 0.017
        elif comp == 'platowg':
            X = 0.725
            Z = 0.
        else:
            X = float(comp.split('z')[1].split('x')[0])/100.
            Z = float(comp.split('z')[0])/100.
    
        Y = 1. - X - Z
    
        for k, m in enumerate(Minit):
    
            for l, r in enumerate(Rinit):
                ! rm /Users/emily/mesa-24.08.1/data/eosDT_data/cache/*planetblend*
                ! rm ./LOGS/*
                ! rm ./photos/*
            
                inlist_create = mh.Inlist('./inlist_create')
                create_modelname = "planet_create_{0}_Mj_{1}_Rj_{2}_{3}.mod".format(m,r,comp,eos)
                create_profilename = "planet_create_{0}_Mj_{1}_Rj_{2}_{3}.profile".format(m,r,comp,eos)
                
                if os.path.isfile(create_modelname):
                    print("create already exists")
                    continue

                
                inlist_create.set_option('save_model_filename', create_modelname)
                inlist_create.set_option('filename_for_profile_when_terminate', create_profilename)
                inlist_create.set_option('mass_in_gm_for_create_initial_model', m*mJtoGrams)
                inlist_create.set_option('radius_in_cm_for_create_initial_model', r*rJtoCm)
        
                inlist_create.set_option('initial_z', Z)
                inlist_create.set_option('initial_y', Y)
                inlist_create.set_option('Zbase', Z)
                inlist_create.set_option('max_age',1.0e4)
        
                inlist_create.set_option('use_other_eos_component',False)
                inlist_create.set_option('use_other_eos_results',False)
        
                inlist_create.set_option('history_columns_file','../plato_benchmarking_history_columns.list')
                inlist_create.set_option('profile_columns_file','../plato_benchmarking_profile_columns.list')

                if eos=='MESA_default':
                    inlist_create.set_option('use_other_eos_component',False)
                    inlist_create.set_option('use_other_eos_results',False)
    
                else:
                    inlist_create.set_option('use_other_eos_component',True)
                    inlist_create.set_option('use_other_eos_results',True)
                    inlist_create.set_option("eos_integer_ctrl(1)",1)
                    inlist_create.set_option("eos_integer_ctrl(2)",1)
                    inlist_create.set_option("eos_logical_ctrl(1)",False)    
                    inlist_create.set_option("eos_character_ctrl(1)",eos_str)
                
                f = open('../rn_create_template', 'r')
                g = f.read()
                f.close()
                #print(g)
                g = g.replace("<<create_model>>", create_modelname)
                        
                h = open('./rn_temp', 'w')
                h.write(g)
                h.close()
                shutil.copyfile('./rn_temp','./rn')
        
                os.system('./mk')
                os.system('./rn')
                

# evolve initial models without relaxing (homogeneous composition only, without setting initial entropy profile)

In [ ]:
clean = True

for i, eos in enumerate(eos_names):
    print(eos)
    this_eos_dir = '/Users/emily/Documents/astro/giant_planets/MESA_EoS/thermo_consistency/MESA_runs/eos_' + eos
    os.chdir(this_eos_dir)
    
    if clean is True:
        ! rm /Users/emily/mesa-24.08.1/data/eosDT_data/cache/*planetblend*
        ! rm /Users/emily/mesa-24.08.1/data/eosDT_data/cache/*CD21+AQUA*

        ! rm ./photos/*
        #! rm ./*.mod
        #! rm ./*.profile
        #! rm ./*.history
        #! rm ./*_consistency.dat
        #! rm ./*.terminationprofile
        #! rm ./LOGS/*

    for j, comp in enumerate(comps):
        
        if comp == 'protosolar':
            X = 0.706
            Z = 0.017
        elif comp == 'platowg':
            X = 0.725
            Z = 0.
        else:
            X = float(comp.split('z')[1].split('x')[0])/100.
            Z = float(comp.split('z')[0])/100.
        
        Y = 1. - X - Z

        print(X,Y,Z)
            
        for k, m in enumerate(Minit):  
            for l, r in enumerate(Rinit):
               
                inlist_evolve = mh.Inlist('./inlist_evolve')
    
                create_modelname = "../make_initial_models/planet_create_{0}_Mj_{1}_Rj_{2}_{3}.mod".format(m,r,comp,eos)
                
                modelname = "planet_{0}Gyr_{1}_Mj_{2}_Rj_{3}.mod".format(agestr,m,r,comp)
                profilename="planet_{0}_Mj_{1}_Rj_{2}.terminationprofile".format(m,r,comp)
                historyname="planet_{0}Gyr_{1}_Mj_{2}_Rj_{3}.history".format(agestr,m,r,comp)
                profileprefix = "planet_{0}Gyr_{1}_Mj_{2}_Rj_{3}_profile".format(agestr,m,r,comp)
    
    
                inlist_evolve.set_option('load_model_filename', create_modelname)
                inlist_evolve.set_option('save_model_filename', modelname)
                inlist_evolve.set_option('filename_for_profile_when_terminate', profilename)
    
                inlist_evolve.set_option('profile_data_prefix',profileprefix)
                
                inlist_evolve.set_option('Zbase', Z)
                inlist_evolve.set_option('max_age',age)
    
                inlist_evolve.set_option('initial_z', Z)
                inlist_evolve.set_option('initial_y', Y)
    
                consistencydataname = "{0}_consistency.dat".format(comp)
    
                print("")
                print("modelname is {0}".format(modelname))
                print("")
    
                if not os.path.isfile(create_modelname):
                    print("no created model; skipping")
                    continue
    
                if os.path.isfile(modelname):
                    print("evolve already succeeded")
                    continue
    
                if eos=='MESA_default':
                    inlist_evolve.set_option('use_other_eos_component',False)
                    inlist_evolve.set_option('use_other_eos_results',False)
    
                else:
                    inlist_evolve.set_option('use_other_eos_component',True)
                    inlist_evolve.set_option('use_other_eos_results',True)
                    inlist_evolve.set_option("eos_integer_ctrl(1)",1)
                    inlist_evolve.set_option("eos_integer_ctrl(2)",1)
                    inlist_evolve.set_option("eos_logical_ctrl(1)",True)
    
                    if 'CD21+AQUA' in eos:
                        eos_str = "mesa-CD21+AQUA_"
                        
                    elif 'TC' in eos or 'control' in eos:
                        eos_name_stem = eos.split("CMS19_")[1] + "_"
                        eos_str = "mesa-planetblend-" + eos_name_stem
                        
                    else:
                        eos_str = "mesa-planetblend_"
        
                    print(eos_str)
                    inlist_evolve.set_option("eos_character_ctrl(1)",eos_str)
    
    
                f = open('../rn_evolve_template', 'r')
                g = f.read()
                f.close()
                #print(g)
                g = g.replace("<<evolve_model>>", modelname)
                    
                h = open('./rn_temp', 'w')
                h.write(g)
                h.close()
                shutil.copyfile('./rn_temp','./rn')
    
    
                
                inlist_evolve.set_option('report_ierr',True)
                inlist_evolve.set_option('report_solver_progress',False)
                inlist_evolve.set_option('solver_check_everything',False)
                inlist_evolve.set_option('mesh_delta_coeff',1.0)
    
                inlist_evolve.set_option('atm_option','T_tau')
                inlist_evolve.set_option('atm_T_tau_relation','Eddington')
                inlist_evolve.set_option('atm_T_tau_opacity','fixed')
    
                inlist_evolve.remove_option('atm_irradiated_T_eq')
                inlist_evolve.remove_option('atm_irradiated_opacity')
                inlist_evolve.remove_option('atm_irradiated_kap_v_div_kap_th')
                inlist_evolve.remove_option('atm_irradiated_max_iters')
                inlist_evolve.remove_option('relax_tau_factor')
                inlist_evolve.remove_option('relax_to_this_tau_factor') 
                
                inlist_evolve.set_option("column_depth_for_irradiation", -1)
                inlist_evolve.set_option("irradiation_flux",0)
                #inlist_evolve.set_option("column_depth_for_irradiation", 300.)
                #inlist_evolve.set_option("irradiation_flux",5.0e4)
                inlist_evolve.set_option('use_other_surface_PT',False)
                inlist_evolve.set_option('scale_max_correction_for_negative_surf_lum',False)
    
                inlist_evolve.set_option('history_columns_file','./history_columns.list')
                inlist_evolve.set_option('profile_columns_file','./profile_columns.list')
                inlist_evolve.set_option('time_delta_coeff',1.0)
                inlist_evolve.set_option('max_years_for_timestep',0)
                inlist_evolve.set_option('terminal_interval',50)
                inlist_evolve.set_option('profile_interval',50)
                inlist_evolve.set_option('max_model_number',250)                
                                
                # execute mk, rn scripts
                os.system('./mk')
                os.system('./rn')
                print("")
                print("")
    
                if eos!='MESA_default':
                    try:
                        shutil.copyfile('./{0}consistency.dat'.format(eos_str), consistencydataname)
                        os.remove("./{0}consistency.dat".format(eos_str))
                    except FileNotFoundError:
                        pass
    
                try:
                    os.system('mv ./LOGS/history.data ./{0}'.format(historyname))
                except FileNotFoundError:
                    pass

# relax composition and entropy & evolve initial models

In [ ]:
clean = True

for i, eos in enumerate(eos_names):
    print(eos)
    this_eos_dir = '/Users/emily/Documents/astro/giant_planets/MESA_EoS/thermo_consistency/MESA_runs/eos_' + eos
    os.chdir(this_eos_dir)
    
    if clean is True:
        ! rm /Users/emily/mesa-24.08.1/data/eosDT_data/cache/*planetblend*
        ! rm /Users/emily/mesa-24.08.1/data/eosDT_data/cache/*CD21+AQUA*

        ! rm ./photos/*
        #! rm ./*.mod
        #! rm ./*.profile
        #! rm ./*.history
        #! rm ./*_consistency.dat
        #! rm ./*.terminationprofile
        #! rm ./LOGS/*

    for j, comp in enumerate(comps):
        
        if comp == 'protosolar':
            X = 0.706
            Z = 0.017
        elif comp == 'platowg':
            X = 0.725
            Z = 0.
        else:
            X = float(comp.split('z')[1].split('x')[0])/100.
            Z = float(comp.split('z')[0])/100.
        
        Y = 1. - X - Z

        print(X,Y,Z)
            
        for k, m in enumerate(Minit):  
            for l, r in enumerate(Rinit):

                for ii, prof in enumerate(comp_profiles):
                    for jj, s in enumerate(Sinit):
                   
                        inlist_relax = mh.Inlist('./inlist_relax')
            
                        create_modelname = "../make_initial_models/planet_create_{0}_Mj_{1}_Rj_{2}_{3}.mod".format(m,r,comp,eos)
                        create_profile_filename = "../make_initial_models/planet_create_{0}_Mj_{1}_Rj_{2}_{3}.profile".format(m,r,comp,eos)
                        
                        relax_modelname = "planet_relax_{0}_Mj_{1}_Rj_{2}_{3}_{4}.mod".format(m,r,comp,prof,s)
                        relax_profilename="planet_relax_{0}_Mj_{1}_Rj_{2}_{3}_{4}.profile".format(m,r,comp,prof,s)
                        relax_composition_filename = "composition_{0}_Mj_{1}_Rj_{2}_{3}_{4}.dat".format(m,r,comp,prof,s)
                        relax_entropy_filename = "entropy_{0}_Mj_{1}_Rj_{2}_{3}_{4}.dat".format(m,r,comp,prof,s)
            
            
                        inlist_relax.set_option('load_model_filename', create_modelname)
                        inlist_relax.set_option('save_model_filename', relax_modelname)
                        inlist_relax.set_option('filename_for_profile_when_terminate', relax_profilename)

                        inlist_relax.set_option('change_net', True)
                        inlist_relax.set_option('new_net_name', 'pp_extras.net')

                        inlist_relax.set_option('set_initial_cumulative_energy_error', True)
                        inlist_relax.set_option('new_cumulative_energy_error', 0.0)
                        inlist_relax.set_option('set_initial_age', True)
                        inlist_relax.set_option('initial_age', 0.0)

                        inlist_relax.set_option('relax_initial_composition', True)
                        inlist_relax.set_option('relax_composition_filename', relax_composition_filename)
                        inlist_relax.set_option('set_initial_dt', True)
                        inlist_relax.set_option('years_for_initial_dt', 1)
                        inlist_relax.set_option('timescale_for_relax_composition', 1.e-3)
                        
                        inlist_relax.set_option('timescale_for_relax_composition',1e-3)
                        inlist_relax.set_option('years_for_initial_dt',1)
                        inlist_relax.set_option('use_gold2_tolerances', False)

                        inlist_relax.set_option('relax_initial_entropy', True)
                        inlist_relax.set_option('relax_entropy_filename', relax_entropy_filename)
                        inlist_relax.set_option('timescale_for_relax_entropy', 1.e-8)
                        inlist_relax.set_option('max_dt_for_relax_entropy', 1.e-9)
                        inlist_relax.set_option('num_timescales_for_relax_entropy', 1000)

                        inlist_relax.set_option('Zbase', Z)
                        inlist_relax.set_option('max_age',1.e4)
            
                        print("")
                        print("relax_modelname is {0}".format(relax_modelname))
                        print("")
            
                        if not os.path.isfile(create_modelname):
                            print("no created model; skipping")
                            continue
            
                        if os.path.isfile(relax_modelname):
                            print("relax already succeeded")
                            continue
                        else:
                            print(relax_modelname)

                        if eos=='MESA_default':
                            inlist_relax.set_option('use_other_eos_component',False)
                            inlist_relax.set_option('use_other_eos_results',False)
            
                        else:
                            inlist_relax.set_option('use_other_eos_component',True)
                            inlist_relax.set_option('use_other_eos_results',True)
                            inlist_relax.set_option("eos_integer_ctrl(1)",1)
                            inlist_relax.set_option("eos_integer_ctrl(2)",1)
                            inlist_relax.set_option("eos_logical_ctrl(1)",False)
            
                            if 'CD21+AQUA' in eos:
                                eos_str = "mesa-CD21+AQUA_"
                                
                            elif 'TC' in eos or 'control' in eos:
                                eos_name_stem = eos.split("CMS19_")[1] + "_"
                                eos_str = "mesa-planetblend-" + eos_name_stem
                                
                            else:
                                eos_str = "mesa-planetblend_"
                
                            print(eos_str)
                            inlist_relax.set_option("eos_character_ctrl(1)",eos_str)
                            
                        inlist_relax.set_option('report_ierr',True)
                        inlist_relax.set_option('report_solver_progress',False)
                        inlist_relax.set_option('solver_check_everything',False)
                        inlist_relax.set_option('mesh_delta_coeff',1.0)
            
                        inlist_relax.set_option('atm_option','T_tau')
                        inlist_relax.set_option('atm_T_tau_relation','Eddington')
                        inlist_relax.set_option('atm_T_tau_opacity','fixed')
                        inlist_relax.set_option('scale_max_correction_for_negative_surf_lum',True)
                    
            
                        inlist_relax.remove_option('atm_irradiated_T_eq')
                        inlist_relax.remove_option('atm_irradiated_opacity')
                        inlist_relax.remove_option('atm_irradiated_kap_v_div_kap_th')
                        inlist_relax.remove_option('atm_irradiated_max_iters')
                        inlist_relax.remove_option('relax_tau_factor')
                        inlist_relax.remove_option('relax_to_this_tau_factor') 
                    
                        
                        inlist_relax.set_option("column_depth_for_irradiation", -1)
                        inlist_relax.set_option("irradiation_flux",0)
                        #inlist_relax.set_option("column_depth_for_irradiation", 300.)
                        #inlist_relax.set_option("irradiation_flux",5.0e4)
                        inlist_relax.set_option('use_other_surface_PT',False)
                        inlist_relax.set_option('scale_max_correction_for_negative_surf_lum',False)
            
                        inlist_relax.set_option('history_columns_file','./history_columns.list')
                        inlist_relax.set_option('profile_columns_file','./profile_columns.list')
                        inlist_relax.set_option('time_delta_coeff',1.0)
                        inlist_relax.set_option('max_years_for_timestep',0)
                        inlist_relax.set_option('terminal_interval',50)
                        inlist_relax.set_option('profile_interval',50)
                        inlist_relax.set_option('max_model_number',250)


                        # initial entropy
                        desired_s_str = str(s)
        
                        # this will be the average metallicity regardless of the functional form of the composition profile 
                        desired_mz = 20.0 # earthMasses
                        desired_mz = np.round(desired_mz, 2)
                        desired_mz_str = '-mz ' + str(desired_mz)
        
                        # zatm, zc, and q1 will depend on the functional form of the composition profile
                        if prof == 'uniform':
                            desired_zc_str = ''
                            desired_zatm_str = ''
                            desired_q1_str = ''
                        elif prof == 'linear':
                            desired_zc_str = ''
                            desired_zatm = 0.017
                            desired_zatm_str = '-zatm ' + str(desired_zatm)
                            desired_q1_str = ''
                        elif prof == 'exponential':
                            desired_zc_str = ''
                            desired_zatm = 0.017
                            desired_zatm_str = '-zatm ' + str(desired_zatm)
                            desired_q1_str = ''
                        elif prof == 'gaussian':
                            desired_zc_str = ''
                            desired_zatm = 0.017
                            desired_zatm_str = '-zatm ' + str(desired_zatm)
                            desired_q1_str = ''
                        elif prof == 'core_uniform':
                            desired_zc_str = ''
                            desired_zatm = 0.017
                            desired_zatm_str = '-zatm ' + str(desired_zatm)
                            desired_q1 = 0.1
                            desired_q1_str = '-q1 ' + str(desired_q1)
                        elif prof == 'core_linear':
                            desired_zc_str = ''
                            desired_zatm = 0.017
                            desired_zatm_str = '-zatm ' + str(desired_zatm)
                            desired_q1 = 0.1
                            desired_q1_str = '-q1 ' + str(desired_q1)
                        elif prof == 'core_exponential':
                            desired_zc_str = ''
                            desired_zatm = 0.017
                            desired_zatm_str = '-zatm ' + str(desired_zatm)
                            desired_q1 = 0.1
                            desired_q1_str = '-q1 ' + str(desired_q1)
                        elif prof == 'core_gaussian':
                            desired_zc_str = ''
                            desired_zatm = 0.017
                            desired_zatm_str = '-zatm ' + str(desired_zatm)
                            desired_q1 = 0.1
                            desired_q1_str = '-q1 ' + str(desired_q1)

                        f = open('../rn_relax_template', 'r')
                        g = f.read()
                        f.close()
                        #print(g)
                        g = g.replace("<<create_profile_name>>", create_profile_filename)
                        g = g.replace("<<functional_form>>", prof)
                        g = g.replace("<<composition_outfile_name>>", relax_composition_filename)
                        g = g.replace("<<entropy_outfile_name>>", relax_entropy_filename)
                        g = g.replace("<<desired_specific_entropy>>", desired_s_str)
                        g = g.replace("<<desired_mz>>", desired_mz_str)
                        g = g.replace("<<desired_zc>>", desired_zc_str)
                        g = g.replace("<<desired_zatm>>", desired_zatm_str)
                        g = g.replace("<<desired_q1>>", desired_q1_str)
                        g = g.replace("<<relax_model_filename>>", relax_modelname)
                            
                        h = open('./rn_temp', 'w')
                        h.write(g)
                        h.close()
                        shutil.copyfile('./rn_temp','./rn')
            
                                        
                        # execute mk, rn scripts
                        os.system('./mk')
                        os.system('./rn')
                        print("")
                        print("")
            


In [ ]:
print((0.017*u.jupiterMass).to(u.earthMass))

In [ ]:
clean = True

for i, eos in enumerate(eos_names):
    print(eos)
    this_eos_dir = '/Users/emily/Documents/astro/giant_planets/MESA_EoS/thermo_consistency/MESA_runs/eos_' + eos
    os.chdir(this_eos_dir)
    
    if clean is True:
        ! rm /Users/emily/mesa-24.08.1/data/eosDT_data/cache/*planetblend*
        ! rm /Users/emily/mesa-24.08.1/data/eosDT_data/cache/*CD21+AQUA*

        ! rm ./photos/*
        #! rm ./*.mod
        #! rm ./*.profile
        #! rm ./*.history
        #! rm ./*_consistency.dat
        #! rm ./*.terminationprofile
        #! rm ./LOGS/*

    for j, comp in enumerate(comps):
        
        if comp == 'protosolar':
            X = 0.706
            Z = 0.017
        elif comp == 'platowg':
            X = 0.725
            Z = 0.
        else:
            X = float(comp.split('z')[1].split('x')[0])/100.
            Z = float(comp.split('z')[0])/100.
        
        Y = 1. - X - Z

        print(X,Y,Z)
            
        for k, m in enumerate(Minit):  
            for l, r in enumerate(Rinit):
                for ii, prof in enumerate(comp_profiles):
                    for jj, s in enumerate(Sinit):
               
                        inlist_evolve = mh.Inlist('./inlist_evolve')
            
                        relax_modelname = "planet_relax_{0}_Mj_{1}_Rj_{2}_{3}_{4}.mod".format(m,r,comp,prof,s)
                        
                        modelname = "planet_{0}Gyr_{1}_Mj_{2}_Rj_{3}_{4}_{5}.mod".format(agestr,m,r,comp,prof,s)
                        profilename="planet_{0}_Mj_{1}_Rj_{2}_{3}_{4}.terminationprofile".format(m,r,comp,prof,s)
                        historyname="planet_{0}Gyr_{1}_Mj_{2}_Rj_{3}_{4}_{5}.history".format(agestr,m,r,comp,prof,s)
                        profileprefix = "planet_{0}Gyr_{1}_Mj_{2}_Rj_{3}_{4}_{5}_profile".format(agestr,m,r,comp,prof,s)
            
            
                        inlist_evolve.set_option('load_model_filename', relax_modelname)
                        inlist_evolve.set_option('save_model_filename', modelname)
                        inlist_evolve.set_option('filename_for_profile_when_terminate', profilename)
            
                        inlist_evolve.set_option('profile_data_prefix',profileprefix)
                        
                        inlist_evolve.set_option('Zbase', Z)
                        inlist_evolve.set_option('max_age',age)
            
                        inlist_evolve.set_option('initial_z', Z)
                        inlist_evolve.set_option('initial_y', Y)
            
                        consistencydataname = "{0}_consistency.dat".format(comp)
            
                        print("")
                        print("modelname is {0}".format(modelname))
                        print("")
            
                        if not os.path.isfile(relax_modelname):
                            print("no relaxed model; skipping")
                            continue
            
                        if os.path.isfile(modelname):
                            print("evolve already succeeded")
                            continue
            
                        if eos=='MESA_default':
                            inlist_evolve.set_option('use_other_eos_component',False)
                            inlist_evolve.set_option('use_other_eos_results',False)
            
                        else:
                            inlist_evolve.set_option('use_other_eos_component',True)
                            inlist_evolve.set_option('use_other_eos_results',True)
                            inlist_evolve.set_option("eos_integer_ctrl(1)",1)
                            inlist_evolve.set_option("eos_integer_ctrl(2)",1)
                            inlist_evolve.set_option("eos_logical_ctrl(1)",True)
            
                            if 'CD21+AQUA' in eos:
                                eos_str = "mesa-CD21+AQUA_"
                                
                            elif 'TC' in eos or 'control' in eos:
                                eos_name_stem = eos.split("CMS19_")[1] + "_"
                                eos_str = "mesa-planetblend-" + eos_name_stem
                                
                            else:
                                eos_str = "mesa-planetblend_"
                
                            print(eos_str)
                            inlist_evolve.set_option("eos_character_ctrl(1)",eos_str)
            
            
                        f = open('../rn_evolve_template', 'r')
                        g = f.read()
                        f.close()
                        #print(g)
                        g = g.replace("<<evolve_model>>", modelname)
                            
                        h = open('./rn_temp', 'w')
                        h.write(g)
                        h.close()
                        shutil.copyfile('./rn_temp','./rn')
            
            
                        
                        inlist_evolve.set_option('report_ierr',True)
                        inlist_evolve.set_option('report_solver_progress',False)
                        inlist_evolve.set_option('solver_check_everything',False)
                        inlist_evolve.set_option('mesh_delta_coeff',1.0)
            
                        inlist_evolve.set_option('atm_option','T_tau')
                        inlist_evolve.set_option('atm_T_tau_relation','Eddington')
                        inlist_evolve.set_option('atm_T_tau_opacity','fixed')
            
                        inlist_evolve.remove_option('atm_irradiated_T_eq')
                        inlist_evolve.remove_option('atm_irradiated_opacity')
                        inlist_evolve.remove_option('atm_irradiated_kap_v_div_kap_th')
                        inlist_evolve.remove_option('atm_irradiated_max_iters')
                        inlist_evolve.remove_option('relax_tau_factor')
                        inlist_evolve.remove_option('relax_to_this_tau_factor') 
                        
                        inlist_evolve.set_option("column_depth_for_irradiation", -1)
                        inlist_evolve.set_option("irradiation_flux",0)
                        #inlist_evolve.set_option("column_depth_for_irradiation", 300.)
                        #inlist_evolve.set_option("irradiation_flux",5.0e4)
                        inlist_evolve.set_option('use_other_surface_PT',False)
                        inlist_evolve.set_option('scale_max_correction_for_negative_surf_lum',False)
            
                        inlist_evolve.set_option('history_columns_file','./history_columns.list')
                        inlist_evolve.set_option('profile_columns_file','./profile_columns.list')
                        inlist_evolve.set_option('time_delta_coeff',1.0)
                        inlist_evolve.set_option('max_years_for_timestep',0)
                        inlist_evolve.set_option('terminal_interval',50)
                        inlist_evolve.set_option('profile_interval',50)
                        inlist_evolve.set_option('max_model_number',250)                
                                        
                        # execute mk, rn scripts
                        os.system('./mk')
                        os.system('./rn')
                        print("")
                        print("")
            
                        if eos!='MESA_default':
                            try:
                                shutil.copyfile('./{0}consistency.dat'.format(eos_str), consistencydataname)
                                os.remove("./{0}consistency.dat".format(eos_str))
                            except FileNotFoundError:
                                pass
            
                        try:
                            os.system('mv ./LOGS/history.data ./{0}'.format(historyname))
                        except FileNotFoundError:
                            pass

# Jupiter-like models for PLATO wg benchmarking

In [ ]:
# eos names
#eos_names = ['MESA_default','CD21+AQUA_orig_meos_implementation','CMS19_orig_meos_implementation','CMS19_TC_s=1','CD21_Y0275']
eos_names = ['CD21_Y0275']
comps = ['platowg']

Minit = np.atleast_1d(np.array((1.0)))
#Rinit = np.atleast_1d(np.array((2.0,5.0)))
Rinit = np.atleast_1d(np.array((2.0)))

age = 5.0e9
agestr = str(np.round(age/1.e9, 1))
print(agestr)

atm = 'customatm'

In [ ]:
clean = True

for i, eos in enumerate(eos_names):
    print(eos)
    this_eos_dir = '/Users/emily/Documents/astro/giant_planets/MESA_EoS/thermo_consistency/MESA_runs/eos_' + eos
    os.chdir(this_eos_dir)
    
    if clean is True:
        ! rm /Users/emily/mesa-24.08.1/data/eosDT_data/cache/*planetblend*
        ! rm /Users/emily/mesa-24.08.1/data/eosDT_data/cache/*CD21+AQUA*
        #! rm ./*irradiatedgrey*
        #! rm ./LOGS/*irradiatedgrey*

        ! rm ./photos/*
        #! rm ./*.mod
        #! rm ./*.profile
        #! rm ./*.history
        #! rm ./*_consistency.dat
        #! rm ./*.terminationprofile
        #! rm ./LOGS/*
        #! rm ./src/data/*
                         
    for j, comp in enumerate(comps):
        
        if comp == 'protosolar':
            X = 0.706
            Z = 0.017
        elif comp == 'platowg':
            X = 0.725
            Z = 0.
        else:
            X = float(comp.split('z')[1].split('x')[0])/100.
            Z = float(comp.split('z')[0])/100.
        
        Y = 1. - X - Z

        print(X,Y,Z)
            
        for k, m in enumerate(Minit):  
            for l, r in enumerate(Rinit):
               
                inlist_evolve = mh.Inlist('./inlist_evolve')
    
                create_modelname = "../make_initial_models/planet_create_{0}_Mj_{1}_Rj_{2}_{3}.mod".format(m,r,comp,eos)
                
                modelname = "planet_{0}Gyr_{1}_Mj_{2}_Rj_{3}_{4}.mod".format(agestr,m,r,comp,atm)
                profilename="planet_{0}_Mj_{1}_Rj_{2}_{3}.terminationprofile".format(m,r,comp,atm)
                historyname="planet_{0}Gyr_{1}_Mj_{2}_Rj_{3}_{4}.history".format(agestr,m,r,comp,atm)
                profileprefix = "planet_{0}Gyr_{1}_Mj_{2}_Rj_{3}_{4}_profile".format(agestr,m,r,comp,atm)
    
    
                inlist_evolve.set_option('load_model_filename', create_modelname)
                inlist_evolve.set_option('save_model_filename', modelname)
                inlist_evolve.set_option('filename_for_profile_when_terminate', profilename)
    
                inlist_evolve.set_option('profile_data_prefix',profileprefix)
                
                inlist_evolve.set_option('Zbase', Z)
                inlist_evolve.set_option('max_age',age)
    
                inlist_evolve.set_option('initial_z', Z)
                inlist_evolve.set_option('initial_y', Y)
                
                consistencydataname = "{0}_consistency.dat".format(comp)
    
                print("")
                print("modelname is {0}".format(modelname))
                print("")
    
                if not os.path.isfile(create_modelname):
                    print("no created model; skipping")
                    continue
    
                #if os.path.isfile(modelname):
                #    print("evolve already succeeded")
                #    continue
    
                if eos=='MESA_default':
                    inlist_evolve.set_option('use_other_eos_component',False)
                    inlist_evolve.set_option('use_other_eos_results',False)
    
                else:
                    inlist_evolve.set_option('use_other_eos_component',True)
                    inlist_evolve.set_option('use_other_eos_results',True)
                    inlist_evolve.set_option("eos_integer_ctrl(1)",1)
                    inlist_evolve.set_option("eos_integer_ctrl(2)",1)
                    inlist_evolve.set_option("eos_logical_ctrl(1)",False)
    
                    if 'CD21' in eos:
                        eos_str = "mesa-CD21+AQUA_"
                        
                    elif 'TC' in eos or 'control' in eos:
                        eos_name_stem = eos.split("CMS19_")[1] + "_"
                        eos_str = "mesa-planetblend-" + eos_name_stem
                        
                    else:
                        eos_str = "mesa-planetblend_"
        
                    print(eos_str)
                    inlist_evolve.set_option("eos_character_ctrl(1)",eos_str)
    
    
                f = open('../rn_evolve_template', 'r')
                g = f.read()
                f.close()
                #print(g)
                g = g.replace("<<evolve_model>>", modelname)
                    
                h = open('./rn_temp', 'w')
                h.write(g)
                h.close()
                shutil.copyfile('./rn_temp','./rn')
    
                inlist_evolve.set_option("column_depth_for_irradiation", -1)
                inlist_evolve.set_option("irradiation_flux",0)
    
                #inlist_evolve.set_option("column_depth_for_irradiation", 300.)
                #inlist_evolve.set_option("irradiation_flux",5.0e4)
    
                inlist_evolve.set_option('report_ierr',True)
                inlist_evolve.set_option('report_solver_progress',False)
                inlist_evolve.set_option('solver_check_everything',False)
                inlist_evolve.set_option('mesh_delta_coeff',0.5)
                inlist_evolve.set_option('history_columns_file','../plato_benchmarking_history_columns.list')
                inlist_evolve.set_option('profile_columns_file','../plato_benchmarking_profile_columns.list')
                inlist_evolve.set_option('time_delta_coeff',1.0)
                inlist_evolve.set_option('max_years_for_timestep',1.e7)
                #inlist_evolve.set_option('max_years_for_timestep',0)
                inlist_evolve.set_option('profile_interval',10)
                #inlist_evolve.set_option('profile_interval',50)
                inlist_evolve.set_option('max_model_number',1000)
                #inlist_evolve.set_option('max_model_number',250)
                inlist_evolve.set_option('relax_tau_factor',True)
                inlist_evolve.set_option('relax_to_this_tau_factor',9)
    
                # Atmospheric controls for Jupiter-like model
                if atm == 'ttau':
                    inlist_evolve.set_option('atm_option','T_tau')
                    inlist_evolve.set_option('atm_T_tau_relation','Eddington')
                    inlist_evolve.set_option('atm_T_tau_opacity','fixed')
        
                    inlist_evolve.remove_option('atm_irradiated_T_eq')
                    inlist_evolve.remove_option('atm_irradiated_opacity')
                    inlist_evolve.remove_option('atm_irradiated_kap_v_div_kap_th')
                    inlist_evolve.remove_option('atm_irradiated_max_iters')
                    inlist_evolve.remove_option('relax_tau_factor')
                    inlist_evolve.remove_option('relax_to_this_tau_factor') 
                    
                    inlist_evolve.set_option("column_depth_for_irradiation", -1)
                    inlist_evolve.set_option("irradiation_flux",0)
                    #inlist_evolve.set_option("column_depth_for_irradiation", 300.)
                    #inlist_evolve.set_option("irradiation_flux",5.0e4)
                    inlist_evolve.set_option('use_other_surface_PT',False)
                    inlist_evolve.set_option('scale_max_correction_for_negative_surf_lum',False)
                    
                if atm == 'irradiatedgrey':
                    inlist_evolve.remove_option('atm_T_tau_relation')
                    inlist_evolve.remove_option('atm_T_tau_opacity')
                    inlist_evolve.set_option("column_depth_for_irradiation", -1)
                    inlist_evolve.set_option("irradiation_flux",0)
                    #inlist_evolve.set_option("column_depth_for_irradiation", 300.)
                    #inlist_evolve.set_option("irradiation_flux",5.0e4)
                    inlist_evolve.set_option('use_other_surface_PT',False)
                    inlist_evolve.set_option('scale_max_correction_for_negative_surf_lum',False)
                    
                    inlist_evolve.set_option('atm_option','irradiated_grey')
                    inlist_evolve.set_option('atm_irradiated_T_eq',110)
                    inlist_evolve.set_option('atm_irradiated_opacity','iterated')
                    inlist_evolve.set_option('atm_irradiated_kap_v_div_kap_th',1)
    
    
                if atm == 'customatm':
                    inlist_evolve.remove_option('atm_T_tau_relation')
                    inlist_evolve.remove_option('atm_T_tau_opacity')
                    
                    inlist_evolve.remove_option('atm_option')
                    inlist_evolve.remove_option('atm_irradiated_T_eq')
                    inlist_evolve.remove_option('atm_irradiated_opacity')
                    inlist_evolve.remove_option('atm_irradiated_kap_v_div_kap_th')
    
                    inlist_evolve.set_option("column_depth_for_irradiation", -1)
                    inlist_evolve.set_option("irradiation_flux",0)
                    #inlist_evolve.set_option("column_depth_for_irradiation", 300.)
                    #inlist_evolve.set_option("irradiation_flux",5.0e4)
    
                    
                    inlist_evolve.set_option('use_other_surface_PT',True)
                    #inlist_evolve.set_option('scale_max_correction_for_negative_surf_lum',True)
    
                # execute mk, rn scripts
                os.system('./mk')
                os.system('./rn')
                print("")
                print("")
    
                if eos!='MESA_default':
                    try:
                        shutil.copyfile('./{0}consistency.dat'.format(eos_str), consistencydataname)
                        os.remove("./{0}consistency.dat".format(eos_str))
                    except FileNotFoundError:
                        pass
    
                try:
                    os.system('mv ./LOGS/history.data ./{0}'.format(historyname))
                except FileNotFoundError:
                    pass
    


# Hot Jupiter models

In [ ]:
# eos names
eos_names = ['MESA_default','CD21+AQUA_orig_meos_implementation','CMS19_orig_meos_implementation','CMS19_TC_s=1','CD21_Y0275']
comps = ['platowg']

Minit = np.atleast_1d(np.array((1.5)))
Rinit = np.atleast_1d(np.array((2.0,5.0)))

age = 5.0e9
agestr = str(np.round(age/1.e9, 1))
print(agestr)

atm = 'irradiatedgrey'

In [ ]:
clean = True

for i, eos in enumerate(eos_names):
    print(eos)
    this_eos_dir = '/Users/emily/Documents/astro/giant_planets/MESA_EoS/thermo_consistency/MESA_runs/eos_' + eos
    os.chdir(this_eos_dir)
    
    if clean is True:
        ! rm /Users/emily/mesa-24.08.1/data/eosDT_data/cache/*planetblend*
        ! rm /Users/emily/mesa-24.08.1/data/eosDT_data/cache/*CD21+AQUA*
        #! rm ./*irradiatedgrey*
        #! rm ./LOGS/*irradiatedgrey*

        ! rm ./photos/*
        #! rm ./*.mod
        #! rm ./*.profile
        #! rm ./*.history
        #! rm ./*_consistency.dat
        #! rm ./*.terminationprofile
        #! rm ./LOGS/*
        #! rm ./src/data/*
                      
    for j, comp in enumerate(comps):
        
        if comp == 'protosolar':
            X = 0.706
            Z = 0.017
        elif comp == 'platowg':
            X = 0.725
            Z = 0.
        else:
            X = float(comp.split('z')[1].split('x')[0])/100.
            Z = float(comp.split('z')[0])/100.
        
        Y = 1. - X - Z

        print(X,Y,Z)
            
        for k, m in enumerate(Minit):  
            for l, r in enumerate(Rinit):
               
                inlist_evolve = mh.Inlist('./inlist_evolve')
    
                create_modelname = "../make_initial_models/planet_create_{0}_Mj_{1}_Rj_{2}_{3}.mod".format(m,r,comp,eos)
                
                modelname = "hj_planet_{0}Gyr_{1}_Mj_{2}_Rj_{3}_{4}.mod".format(agestr,m,r,comp,atm)
                profilename="hj_planet_{0}_Mj_{1}_Rj_{2}_{3}.terminationprofile".format(m,r,comp,atm)
                historyname="hj_planet_{0}Gyr_{1}_Mj_{2}_Rj_{3}_{4}.history".format(agestr,m,r,comp,atm)
                profileprefix = "hj_planet_{0}Gyr_{1}_Mj_{2}_Rj_{3}_{4}_profile".format(agestr,m,r,comp,atm)
    
    
                inlist_evolve.set_option('load_model_filename', create_modelname)
                inlist_evolve.set_option('save_model_filename', modelname)
                inlist_evolve.set_option('filename_for_profile_when_terminate', profilename)
    
                inlist_evolve.set_option('profile_data_prefix',profileprefix)
                
                inlist_evolve.set_option('Zbase', Z)
                inlist_evolve.set_option('max_age',age)
    
                inlist_evolve.set_option('initial_z', Z)
                inlist_evolve.set_option('initial_y', Y)
                
                consistencydataname = "{0}_consistency.dat".format(comp)
    
                print("")
                print("modelname is {0}".format(modelname))
                print("")
    
                if not os.path.isfile(create_modelname):
                    print("no created model; skipping")
                    continue
    
                if os.path.isfile(modelname):
                    print("evolve already succeeded")
                    continue
    
                if eos=='MESA_default':
                    inlist_evolve.set_option('use_other_eos_component',False)
                    inlist_evolve.set_option('use_other_eos_results',False)
    
                else:
                    inlist_evolve.set_option('use_other_eos_component',True)
                    inlist_evolve.set_option('use_other_eos_results',True)
                    inlist_evolve.set_option("eos_integer_ctrl(1)",1)
                    inlist_evolve.set_option("eos_integer_ctrl(2)",1)
                    inlist_evolve.set_option("eos_logical_ctrl(1)",False)
    
                    if 'CD21' in eos:
                        eos_str = "mesa-CD21+AQUA_"
                        
                    elif 'TC' in eos or 'control' in eos:
                        eos_name_stem = eos.split("CMS19_")[1] + "_"
                        eos_str = "mesa-planetblend-" + eos_name_stem
                        
                    else:
                        eos_str = "mesa-planetblend_"
        
                    print(eos_str)
                    inlist_evolve.set_option("eos_character_ctrl(1)",eos_str)
    
    
                f = open('../rn_evolve_template', 'r')
                g = f.read()
                f.close()
                #print(g)
                g = g.replace("<<evolve_model>>", modelname)
                    
                h = open('./rn_temp', 'w')
                h.write(g)
                h.close()
                shutil.copyfile('./rn_temp','./rn')
    
                inlist_evolve.set_option("column_depth_for_irradiation", -1)
                inlist_evolve.set_option("irradiation_flux",0)
    
                #inlist_evolve.set_option("column_depth_for_irradiation", 300.)
                #inlist_evolve.set_option("irradiation_flux",5.0e4)
    
                inlist_evolve.set_option('report_ierr',True)
                inlist_evolve.set_option('report_solver_progress',False)
                inlist_evolve.set_option('solver_check_everything',False)
                inlist_evolve.set_option('mesh_delta_coeff',0.5)
                inlist_evolve.set_option('history_columns_file','../plato_benchmarking_history_columns.list')
                inlist_evolve.set_option('profile_columns_file','../plato_benchmarking_profile_columns.list')
                inlist_evolve.set_option('time_delta_coeff',1.0)
                inlist_evolve.set_option('max_years_for_timestep',1.e7)
                #inlist_evolve.set_option('max_years_for_timestep',0)
                inlist_evolve.set_option('profile_interval',10)
                #inlist_evolve.set_option('profile_interval',50)
                inlist_evolve.set_option('max_model_number',1000)
                #inlist_evolve.set_option('max_model_number',250)
                
                # the below is VERY slow
                inlist_evolve.set_option('relax_tau_factor',True)
                inlist_evolve.set_option('relax_to_this_tau_factor',9)

                inlist_evolve.set_option('initial_model_relax_num_steps',50)
    
                # Atmospheric controls for Jupiter-like model
                if atm == 'ttau':
                    inlist_evolve.set_option('atm_option','T_tau')
                    inlist_evolve.set_option('atm_T_tau_relation','Eddington')
                    inlist_evolve.set_option('atm_T_tau_opacity','fixed')
        
                    inlist_evolve.remove_option('atm_irradiated_T_eq')
                    inlist_evolve.remove_option('atm_irradiated_opacity')
                    inlist_evolve.remove_option('atm_irradiated_kap_v_div_kap_th')
                    inlist_evolve.remove_option('atm_irradiated_max_iters')
                    inlist_evolve.remove_option('relax_tau_factor')
                    inlist_evolve.remove_option('relax_to_this_tau_factor') 
                    
                    inlist_evolve.set_option("column_depth_for_irradiation", -1)
                    inlist_evolve.set_option("irradiation_flux",0)
                    #inlist_evolve.set_option("column_depth_for_irradiation", 300.)
                    #inlist_evolve.set_option("irradiation_flux",5.0e4)
                    inlist_evolve.set_option('use_other_surface_PT',False)
                    inlist_evolve.set_option('scale_max_correction_for_negative_surf_lum',False)
                    
                if atm == 'irradiatedgrey':
                    inlist_evolve.remove_option('atm_T_tau_relation')
                    inlist_evolve.remove_option('atm_T_tau_opacity')
                    inlist_evolve.set_option("column_depth_for_irradiation", -1)
                    inlist_evolve.set_option("irradiation_flux",0)
                    #inlist_evolve.set_option("column_depth_for_irradiation", 300.)
                    #inlist_evolve.set_option("irradiation_flux",5.0e4)
                    inlist_evolve.set_option('use_other_surface_PT',False)
                    inlist_evolve.set_option('scale_max_correction_for_negative_surf_lum',True)
                    
                    inlist_evolve.set_option('atm_option','irradiated_grey')
                    inlist_evolve.set_option('atm_irradiated_T_eq',1000)
                    inlist_evolve.set_option('atm_irradiated_opacity','fixed')
                    inlist_evolve.set_option('atm_irradiated_kap_v',6.e-3)
                    inlist_evolve.set_option('atm_irradiated_kap_v_div_kap_th',0.6)
    
    
                if atm == 'customatm':
                    inlist_evolve.remove_option('atm_T_tau_relation')
                    inlist_evolve.remove_option('atm_T_tau_opacity')
                    
                    inlist_evolve.remove_option('atm_option')
                    inlist_evolve.remove_option('atm_irradiated_T_eq')
                    inlist_evolve.remove_option('atm_irradiated_opacity')
                    inlist_evolve.remove_option('atm_irradiated_kap_v_div_kap_th')
    
                    inlist_evolve.set_option("column_depth_for_irradiation", -1)
                    inlist_evolve.set_option("irradiation_flux",0)
                    #inlist_evolve.set_option("column_depth_for_irradiation", 300.)
                    #inlist_evolve.set_option("irradiation_flux",5.0e4)
    
                    
                    inlist_evolve.set_option('use_other_surface_PT',True)
                    #inlist_evolve.set_option('scale_max_correction_for_negative_surf_lum',True)
    
                # execute mk, rn scripts
                os.system('./mk')
                os.system('./rn')
                print("")
                print("")
    
                if eos!='MESA_default':
                    try:
                        shutil.copyfile('./{0}consistency.dat'.format(eos_str), consistencydataname)
                        os.remove("./{0}consistency.dat".format(eos_str))
                    except FileNotFoundError:
                        pass
    
                try:
                    os.system('mv ./LOGS/history.data ./{0}'.format(historyname))
                except FileNotFoundError:
                    pass
    
